In [1]:
import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

Using device: mps


In [2]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)
model.eval()

similarity_threshold = 0.80

print(f"Loaded model: {model_name}")
print(f"Similarity threshold: {similarity_threshold}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model: sentence-transformers/all-MiniLM-L6-v2
Similarity threshold: 0.8


In [3]:
dataset = load_dataset("glue", "mrpc", split="validation")
max_examples = 300
if max_examples is not None:
    dataset = dataset.select(range(min(max_examples, len(dataset))))

print("Dataset split: glue/mrpc validation")
print(f"Number of examples used: {len(dataset)}")
print("Example row:")
print(dataset[0])

Dataset split: glue/mrpc validation
Number of examples used: 300
Example row:
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}


In [4]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

def encode_texts(texts, batch_size=64, max_length=128):
    all_embeddings = []
    for start_idx in range(0, len(texts), batch_size):
        batch_texts = texts[start_idx:start_idx + batch_size]
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
            embeddings = mean_pool(outputs.last_hidden_state, inputs["attention_mask"])
            embeddings = F.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu())
    return torch.cat(all_embeddings, dim=0)

sentence1_list = dataset["sentence1"]
sentence2_list = dataset["sentence2"]
labels = np.array(dataset["label"])

emb1 = encode_texts(sentence1_list)
emb2 = encode_texts(sentence2_list)
similarities = F.cosine_similarity(emb1, emb2).numpy()
predictions = (similarities >= similarity_threshold).astype(int)

print(f"Completed embedding inference for {len(similarities)} examples.")

Completed embedding inference for 300 examples.


In [5]:
accuracy = accuracy_score(labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary", zero_division=0)
cm = confusion_matrix(labels, predictions)

sim_min = float(np.min(similarities))
sim_max = float(np.max(similarities))
sim_mean = float(np.mean(similarities))
sim_median = float(np.median(similarities))
sim_std = float(np.std(similarities))

pos_mask = labels == 1
neg_mask = labels == 0
pos_mean = float(np.mean(similarities[pos_mask])) if np.any(pos_mask) else float("nan")
neg_mean = float(np.mean(similarities[neg_mask])) if np.any(neg_mask) else float("nan")

print("Evaluation metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print("Confusion matrix:")
print(cm)

print("\nSimilarity summary statistics:")
print(f"min              : {sim_min:.4f}")
print(f"max              : {sim_max:.4f}")
print(f"mean             : {sim_mean:.4f}")
print(f"median           : {sim_median:.4f}")
print(f"std              : {sim_std:.4f}")
print(f"mean(label=1)    : {pos_mean:.4f}")
print(f"mean(label=0)    : {neg_mean:.4f}")

Evaluation metrics:
Accuracy : 0.6567
Precision: 0.8118
Recall   : 0.6603
F1       : 0.7282
Confusion matrix:
[[ 59  32]
 [ 71 138]]

Similarity summary statistics:
min              : 0.3012
max              : 0.9971
mean             : 0.7942
median           : 0.8341
std              : 0.1480
mean(label=1)    : 0.8301
mean(label=0)    : 0.7116


In [6]:
distance_to_threshold = np.abs(similarities - similarity_threshold)
sorted_indices = np.argsort(distance_to_threshold)
top_k = 5
nearest_indices = sorted_indices[:top_k]

label_map = {0: "not_paraphrase", 1: "paraphrase"}

def print_example(idx, title):
    row = dataset[int(idx)]
    true_label = int(labels[idx])
    pred_label = int(predictions[idx])
    sim = float(similarities[idx])
    dist = float(distance_to_threshold[idx])
    print(title)
    print(f"index        : {idx}")
    print(f"sentence1    : {row['sentence1']}")
    print(f"sentence2    : {row['sentence2']}")
    print(f"true label   : {true_label} ({label_map[true_label]})")
    print(f"pred label   : {pred_label} ({label_map[pred_label]})")
    print(f"similarity   : {sim:.4f}")
    print(f"threshold    : {similarity_threshold:.4f}")
    print(f"abs distance : {dist:.4f}")
    print("-" * 80)

print(f"Nearest-to-threshold examples (top {top_k}):")
for rank, idx in enumerate(nearest_indices, start=1):
    print_example(idx, f"Example rank {rank}")

Nearest-to-threshold examples (top 5):
Example rank 1
index        : 232
sentence1    : Leon Williams ' body was found inside his third-floor apartment at 196 Bay St. , in Tompkinsville .
sentence2    : The dead man , Leon Williams , was found in his third-floor apartment .
true label   : 1 (paraphrase)
pred label   : 0 (not_paraphrase)
similarity   : 0.8000
threshold    : 0.8000
abs distance : 0.0000
--------------------------------------------------------------------------------
Example rank 2
index        : 63
sentence1    : That would be a potential setback to Chief Executive Phil Condit 's strategy of bolstering defense-related sales during a slump in jetliner deliveries .
sentence2    : The inquiry may hinder Chief Executive Phil Condit 's strategy of bolstering defense-related sales during a slump in jetliner deliveries .
true label   : 1 (paraphrase)
pred label   : 0 (not_paraphrase)
similarity   : 0.7992
threshold    : 0.8000
abs distance : 0.0008
-----------------------------

In [7]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("dataset_split=glue/mrpc validation")
print(f"device={device}")
print(f"num_examples={len(dataset)}")
print(f"threshold={similarity_threshold:.4f}")
print(f"accuracy={accuracy:.4f}")
print(f"precision={precision:.4f}")
print(f"recall={recall:.4f}")
print(f"f1={f1:.4f}")
print(f"similarity_mean={sim_mean:.4f}")
print(f"similarity_median={sim_median:.4f}")
print(f"similarity_std={sim_std:.4f}")

RESULT SUMMARY
model=sentence-transformers/all-MiniLM-L6-v2
dataset_split=glue/mrpc validation
device=mps
num_examples=300
threshold=0.8000
accuracy=0.6567
precision=0.8118
recall=0.6603
f1=0.7282
similarity_mean=0.7942
similarity_median=0.8341
similarity_std=0.1480
